In [1]:
import pandas as pd
import numpy as np
from gensim.scripts.glove2word2vec import glove2word2vec
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import time
import os

# ───────────────────────────────────────────
# 1. Load your CSV
# ───────────────────────────────────────────
# file_name = "case_study_circuit_patched_Inference_Circuit_New_Baseline1_Qwen3_BATCH8.csv"
base_path = "../../../../"
goto_folder = "ResultGroup/3.Circuit/"
filename = "Circuit-QuantumVLM-3VL-32B-v2814-4bit.csv"
csv_path = f"{base_path}{goto_folder}{filename}"
df = pd.read_csv(csv_path)

# ───────────────────────────────────────────
# 2. Load or convert GloVe
# ───────────────────────────────────────────
glove_txt = "glove.6B.300d.txt"
glove_w2v = "glove.6B.300d.w2v"

if not os.path.exists(glove_w2v):
    print("[INFO] Converting GloVe format to word2vec format...")
    glove2word2vec(glove_txt, glove_w2v)
    print("[INFO] Conversion complete.")
else:
    print("[INFO] word2vec file already exists. Skipping conversion.")

print("[INFO] Loading embeddings...")
model = KeyedVectors.load_word2vec_format(glove_w2v, binary=False)
print("[INFO] Embeddings loaded.")

# ───────────────────────────────────────────
# 3. Helper: Convert a sentence → vector
# ───────────────────────────────────────────
def sentence_vector(text):
    words = str(text).lower().split()
    vecs = [model[w] for w in words if w in model]
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

# ───────────────────────────────────────────
# 4. Seven prediction/output pairs
# ───────────────────────────────────────────
pairs = [
    ("prediction_1", "output_1"),
    ("prediction_2", "output_2"),
    ("prediction_3", "output_3"),
    ("prediction_4", "output_4"),
]

total = len(df)

# ───────────────────────────────────────────
# 5. Compute similarity for each pair
# ───────────────────────────────────────────
for pred_col, gt_col in pairs:
    print(f"\n[INFO] Scoring {pred_col} vs {gt_col} ...")

    scores = []
    last_print = time.time()

    for i, row in df.iterrows():
        v1 = sentence_vector(row[pred_col])
        v2 = sentence_vector(row[gt_col])
        sim = cosine_similarity([v1], [v2])[0][0]
        scores.append(sim)

        if time.time() - last_print > 1:
            pct = (i + 1) / total * 100
            print(f"[INFO] {pred_col}: {i+1}/{total} ({pct:.2f}%)")
            last_print = time.time()

    df[f"glove_score_{pred_col}"] = scores

print("\n[INFO] All scoring finished.")

# ───────────────────────────────────────────
# 6. Save
# ───────────────────────────────────────────
out_path = f"output/3-4-3-1.glove-circuit-baseline-1-{filename}"
df.to_csv(out_path, index=False)

print(f"[INFO] Saved to {out_path}")

# calculate all the mean
mean_scores = {}
total_mean = 0
for pred_col, _ in pairs:
    mean_score = df[f"glove_score_{pred_col}"].mean()
    mean_scores[pred_col] = mean_score
    total_mean += mean_score
    print(f"Mean GloVe score for {pred_col}: {mean_score:.4f}")
    
total_mean /= len(pairs)
print(f"Overall Mean GloVe score: {total_mean:.4f}")
    
# ─────────────────────────────────────────────
# 7. Display mean scores
# ─────────────────────────────────────────────
print("\n[INFO] Mean GloVe Similarity Scores:")
for pred_col, mean_score in mean_scores.items():
    print(f"{pred_col}: {mean_score:.4f}")
# ─────────────────────────────────────────────
# 8. End of script

[INFO] word2vec file already exists. Skipping conversion.
[INFO] Loading embeddings...
[INFO] Embeddings loaded.

[INFO] Scoring prediction_1 vs output_1 ...

[INFO] Scoring prediction_2 vs output_2 ...

[INFO] Scoring prediction_3 vs output_3 ...

[INFO] Scoring prediction_4 vs output_4 ...

[INFO] All scoring finished.
[INFO] Saved to output/3-4-3-1.glove-circuit-baseline-1-Circuit-QuantumVLM-3VL-32B-v2814-4bit.csv
Mean GloVe score for prediction_1: 0.9300
Mean GloVe score for prediction_2: 0.9080
Mean GloVe score for prediction_3: 0.8369
Mean GloVe score for prediction_4: 0.7547
Overall Mean GloVe score: 0.8574

[INFO] Mean GloVe Similarity Scores:
prediction_1: 0.9300
prediction_2: 0.9080
prediction_3: 0.8369
prediction_4: 0.7547
